# 02 - LangChain y chunking

Objetivo: cargar documentos y dividirlos en chunks razonables antes de vectorizar.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from markitdown import MarkItDown

load_dotenv(Path("..").resolve() / ".env")

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))
DOCS_DIR = Path("..").resolve() / "docs"

print({
    "DOCS_DIR": str(DOCS_DIR),
    "CHUNK_SIZE": CHUNK_SIZE,
    "CHUNK_OVERLAP": CHUNK_OVERLAP,
})


## Cargar Markdown ya preparado

Empezamos por el caso mas sencillo: documentos que ya estan en `.md` dentro de `docs/`.


In [ ]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()
print(f"Documentos cargados: {len(documents)}")
documents[0]


## Convertir otros formatos a Markdown con `markitdown`

Cuando los documentos de origen no vienen en Markdown, una opcion muy comoda es convertirlos antes a texto estructurado. `markitdown` sirve para formatos como `pdf`, `docx`, `pptx`, `xlsx` o `html`.

La idea es sencilla:

1. localizar ficheros de otros tipos,
2. convertir cada fichero a Markdown,
3. envolver el resultado en `Document` de LangChain,
4. chunkear todo igual que si hubiera nacido en `.md`.


In [ ]:
markitdown = MarkItDown()
other_source_extensions = [".pdf", ".docx", ".pptx", ".xlsx", ".html"]

other_source_files = [
    path
    for path in DOCS_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in other_source_extensions
]

converted_documents = []
for path in other_source_files:
    result = markitdown.convert(str(path))
    converted_documents.append(
        Document(
            page_content=result.text_content,
            metadata={
                "source": str(path),
                "converted_with": "markitdown",
                "original_extension": path.suffix.lower(),
            },
        )
    )

print({
    "ficheros_encontrados": len(other_source_files),
    "documentos_convertidos": len(converted_documents),
})

if converted_documents:
    converted_documents[0]
else:
    print("No hay todavia archivos PDF, DOCX, PPTX, XLSX o HTML dentro de docs/.")


In [ ]:
all_documents = documents + converted_documents
print(f"Documentos totales para chunking: {len(all_documents)}")


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(all_documents)
print(f"Chunks generados: {len(chunks)}")

for idx, chunk in enumerate(chunks[:3]):
    chunk.metadata["chunk_id"] = idx
    print("=" * 80)
    print(chunk.metadata.get("source", "sin source"))
    print(chunk.page_content[:400])


In [ ]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]
print({
    "min": min(chunk_lengths),
    "max": max(chunk_lengths),
    "media_aprox": round(sum(chunk_lengths) / len(chunk_lengths), 1),
})


Prueba sugerida:

1. Cambia `CHUNK_SIZE` o `CHUNK_OVERLAP` en `.env`.
2. Vuelve a ejecutar este notebook.
3. Observa si cambian los trozos que luego irian a embeddings.
